# Week 8 Coding Practice: Working with Time Series in pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/w08-cp-01-colab-link/module08/week8_time_series_practice.ipynb)

**Estimated time:** 60–75 minutes
**Format:** ungraded guided practice

This activity uses a local bicycle-count dataset to practice working with time-indexed pandas data. Instead of memorizing pandas syntax, focus on how the data are organized, how the time axis changes the analysis, and how different summaries answer different questions.

## Learning objectives

By the end of the activity, you should be able to:

1. load a time-indexed dataset and parse dates correctly;
2. inspect a DatetimeIndex and check for missing or irregular observations;
3. select observations by date and date ranges;
4. aggregate time series to larger frequencies and compare the results;
5. use rolling windows, lagged values, and weekday/weekend comparisons to describe patterns; and
6. evaluate a simple chronological forecast baseline with MAE and RMSE.

## 1. Load and inspect the data

The dataset contains daily bicycle counts on the Fremont Bridge, recorded as a count per day. We will work with a local copy so the notebook runs in a fresh environment without a fragile network download.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

candidate_paths = [
    Path('appliedDataScience/module08/fremont_bridge_daily.csv'),
    Path('fremont_bridge_daily.csv'),
    Path.cwd() / 'appliedDataScience/module08/fremont_bridge_daily.csv',
]
data_path = next((p for p in candidate_paths if p.exists()), candidate_paths[0])
if not data_path.exists():
    raise FileNotFoundError(f'Missing dataset: {data_path}')

bridge = pd.read_csv(data_path, parse_dates=['date'])
bridge = bridge.sort_values('date').reset_index(drop=True)
bridge = bridge.rename(columns={'bikes': 'daily_count'})
bridge = bridge.set_index('date')

print(bridge.head())
print(f'Rows: {len(bridge)}')
print(f'Start: {bridge.index.min()}')
print(f'End: {bridge.index.max()}')
print(f'NaN values: {bridge.isna().sum().sum()}')


### Reasoning checkpoint

Before running the next cell, predict whether counts are likely to be higher on weekdays or weekends, and why. Write one sentence explaining your expectation.

In [ ]:
bridge_daily = bridge.asfreq('D')
print('Inferred frequency:', bridge_daily.index.inferred_freq)
print('Missing dates:', bridge_daily['daily_count'].isna().sum())
print(bridge_daily.head())


## 2. Select observations by date

A time series is not just a list of values; the index tells us when each observation belongs. Use date-based indexing to isolate the time period you care about.

In [ ]:
subset = bridge_daily.loc['2023-01-01':'2023-01-31']
print(subset.head())
print(f'Count in January 2023: {len(subset)} days')
print(f'Mean January 2023 count: {subset["daily_count"].mean():.1f}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
bridge_daily['daily_count'].plot(ax=ax, linewidth=1.2, color='#2c7fb8')
ax.set(title='Daily bicycle counts', xlabel='Date', ylabel='Count')
plt.tight_layout()
plt.show()


### Interpret the time plot

What broad features do you notice? Do the counts appear to rise, fall, or stay roughly steady? Is there evidence of regular repeating patterns?

In [ ]:
rolling_7 = bridge_daily['daily_count'].rolling(window=7, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(12, 4))
bridge_daily['daily_count'].plot(ax=ax, label='Daily count', alpha=0.7, color='#2c7fb8')
rolling_7.plot(ax=ax, label='7-day rolling mean', color='#d95f02', linewidth=2.5)
ax.set(title='Seven-day summary', xlabel='Date', ylabel='Count')
ax.legend()
plt.tight_layout()
plt.show()


## 3. Aggregate and compare time frequencies

Changing the frequency changes the question. Daily counts are useful for detail; weekly totals help us compare a longer cycle.

In [ ]:
weekly_total = bridge_daily['daily_count'].resample('W-MON').sum()
monthly_average = bridge_daily['daily_count'].resample('MS').mean()
print("Weekly totals head:")
print(weekly_total.head())
print("Monthly average head:")
print(monthly_average.head())


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
bridge_daily['daily_count'].plot(ax=axes[0], color='#2c7fb8', alpha=0.8)
axes[0].set(title='Daily counts', ylabel='Count')
weekly_total.plot(ax=axes[1], color='#d95f02', linewidth=2)
axes[1].set(title='Weekly total counts', xlabel='Date', ylabel='Count')
plt.tight_layout()
plt.show()


### Compare weekdays and weekends

A weekend day may have a different pattern from a weekday because commutes, errands, and leisure travel all differ across the week.

In [ ]:
weekday_means = bridge_daily.groupby(bridge_daily.index.day_name())['daily_count'].mean()
ordered_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_means = weekday_means.reindex(ordered_days)
print(weekday_means)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
weekday_means.plot(kind='bar', color='#66a61e', ax=ax)
ax.set(title='Mean daily count by weekday', xlabel='Day of week', ylabel='Average count')
plt.tight_layout()
plt.show()


## 4. Explore lagged observations

A lag compares today's observation to a previous time point. A 1-day lag captures short-run persistence; a 7-day lag captures a weekly pattern.

In [ ]:
bridge_daily['lag_1'] = bridge_daily['daily_count'].shift(1)
bridge_daily['lag_7'] = bridge_daily['daily_count'].shift(7)
bridge_daily[['daily_count', 'lag_1', 'lag_7']].head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(bridge_daily['lag_1'], bridge_daily['daily_count'], alpha=0.25, color='#7570b3')
axes[0].set(title='Lag 1', xlabel='Previous day count', ylabel='Current day count')
axes[1].scatter(bridge_daily['lag_7'], bridge_daily['daily_count'], alpha=0.25, color='#e7298a')
axes[1].set(title='Lag 7', xlabel='Previous week count', ylabel='Current day count')
for ax in axes:
    ax.axline((0, 0), slope=1, color='black', linewidth=1, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


### Reasoning checkpoint

Which lag appears more informative for a typical weekly pattern: 1 day or 7 days? Explain your answer using the plot and the calendar structure of the data.

## 5. A simple forecast baseline

The goal here is not to build an advanced forecast. We will compare a naive baseline and a seasonal-naive baseline using a chronological train/test split.

In [ ]:
train = bridge_daily.iloc[:-28].copy()
test = bridge_daily.iloc[-28:].copy()
naive = pd.Series(train['daily_count'].iloc[-1], index=test.index, name='naive')
seasonal_pattern = train['daily_count'].tail(7).to_numpy()
seasonal_naive = pd.Series(
    np.tile(seasonal_pattern, int(np.ceil(len(test) / 7)) + 1)[: len(test)],
    index=test.index,
    name='seasonal_naive',
)
forecast_df = pd.concat([test['daily_count'], naive, seasonal_naive], axis=1)
forecast_df.columns = ['actual', 'naive', 'seasonal_naive']
print(forecast_df.head())


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_naive = mean_absolute_error(forecast_df['actual'], forecast_df['naive'])
rmse_naive = np.sqrt(mean_squared_error(forecast_df['actual'], forecast_df['naive']))
mae_seasonal = mean_absolute_error(forecast_df['actual'], forecast_df['seasonal_naive'])
rmse_seasonal = np.sqrt(mean_squared_error(forecast_df['actual'], forecast_df['seasonal_naive']))

print(f'Naive MAE: {mae_naive:.1f}')
print(f'Naive RMSE: {rmse_naive:.1f}')
print(f'Seasonal naive MAE: {mae_seasonal:.1f}')
print(f'Seasonal naive RMSE: {rmse_seasonal:.1f}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
forecast_df['actual'].plot(ax=ax, label='Actual', color='#1b9e77', linewidth=1.5)
forecast_df['naive'].plot(ax=ax, label='Naive', linestyle='--', color='#d95f02')
forecast_df['seasonal_naive'].plot(ax=ax, label='Seasonal naive', linestyle=':', color='#7570b3')
ax.set(title='Chronological forecast comparison', xlabel='Date', ylabel='Daily count')
ax.legend()
plt.tight_layout()
plt.show()


### Final check

Which baseline does better on this time period, and why might a seasonal baseline outperform a naive baseline when the data have a weekly pattern? Explain in 2–3 sentences.

## Takeaway

Time series analysis is not just about values in a table. The time index matters. Daily counts can be summarized, compared, and forecast using the right pandas operations. A good analysis asks whether a pattern is real, whether the aggregation matches the decision, and whether the evaluation respects chronology.